In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from google.colab import drive

print("All libraries loaded successfully")

All libraries loaded successfully


In [3]:
# ── Mount Google Drive ─────────────────────────────
drive.mount('/content/drive')

# ── Set paths ──────────────────────────────────────
BASE_PATH     = "/content/drive/MyDrive/thesis_data/Original Reddit Data"
LABELLED_PATH = os.path.join(BASE_PATH, "Labelled Data")
RAW_PATH      = os.path.join(BASE_PATH, "raw data")

print("Base path exists:     ", os.path.exists(BASE_PATH))
print("Labelled path exists: ", os.path.exists(LABELLED_PATH))
print("Raw path exists:      ", os.path.exists(RAW_PATH))

Mounted at /content/drive
Base path exists:      True
Labelled path exists:  True
Raw path exists:       True


In [10]:
# ── Load each labelled file with its category ──────
label_files = {
    "Drug and Alcohol" : os.path.join(LABELLED_PATH, "LD DA 1.csv"),
    "Early Life"       : os.path.join(LABELLED_PATH, "LD EL1.csv"),
    "Personality"      : os.path.join(LABELLED_PATH, "LD PF1.csv"),
    "Trauma and Stress": os.path.join(LABELLED_PATH, "LD TS 1.csv")
}

dfs_b = []
for label, filepath in label_files.items():
    try:
        temp = pd.read_csv(filepath)
        temp['Label'] = label
        dfs_b.append(temp)
        print(f"✓ {label}: {temp.shape[0]} rows | columns: {temp.columns.tolist()}")
    except Exception as e:
        print(f"✗ Failed {label}: {e}")

df_b = pd.concat(dfs_b, ignore_index=True)
print(f"\nPart B total shape: {df_b.shape}")
print("\nLabel distribution:")
print(df_b['Label'].value_counts())

✓ Drug and Alcohol: 223 rows | columns: ['score', 'selftext', 'subreddit', 'title', 'Label']
✓ Early Life: 200 rows | columns: ['score', 'selftext', 'subreddit', 'title', 'Label', 'CAT 1']
✓ Personality: 200 rows | columns: ['score', 'selftext', 'subreddit', 'title', 'Label', 'CAT 1']
✓ Trauma and Stress: 200 rows | columns: ['score', 'selftext', 'subreddit', 'title', 'Label', 'CAT 1']

Part B total shape: (823, 6)

Label distribution:
Label
Drug and Alcohol     223
Early Life           200
Personality          200
Trauma and Stress    200
Name: count, dtype: int64


In [9]:
# ── See what is inside each year folder ────────────
for year in ['2019', '2020', '2021', '2022']:
    year_path = os.path.join(RAW_PATH, year)
    if os.path.exists(year_path):
        files = os.listdir(year_path)
        print(f"\n{year} — {len(files)} files:")
        for f in files[:5]:
            print(f"   {f}")


2019 — 12 files:
   apr
   feb
   JAN
   Aug
   june

2020 — 12 files:
   Apr 20
   Dec 20
   Feb20
   Aug 20
   Mar 20

2021 — 12 files:
   Aug 21
   Apr 21
   Dec 21
   Jul 21
   June 21

2022 — 8 files:
   Feb22
   Aug 22
   Jan 22
   Apr 22
   June 22


In [11]:
# ── Load all CSV files from all years ──────────────
all_files = glob.glob(
    os.path.join(RAW_PATH, "**", "*.csv"), 
    recursive=True
)
print(f"Total CSV files found: {len(all_files)}")

dfs_a = []
failed = []

for filepath in all_files:
    try:
        temp = pd.read_csv(filepath, low_memory=False)
        dfs_a.append(temp)
    except Exception as e:
        failed.append(filepath)
        print(f"✗ Failed: {os.path.basename(filepath)} — {e}")

print(f"\nSuccessfully loaded: {len(dfs_a)} files")
print(f"Failed:             {len(failed)} files")

df_a = pd.concat(dfs_a, ignore_index=True)
print(f"\nPart A total shape: {df_a.shape}")
print("\nColumns:", df_a.columns.tolist())

Total CSV files found: 219

Successfully loaded: 219 files
Failed:             0 files

Part A total shape: (1851580, 8)

Columns: ['Unnamed: 0', 'author', 'created_utc', 'score', 'selftext', 'subreddit', 'title', 'timestamp']


In [12]:
# ── Part A exploration ─────────────────────────────
print("=== MISSING VALUES ===")
print(df_a.isnull().sum())

print("\n=== SUBREDDIT DISTRIBUTION ===")
print(df_a['subreddit'].value_counts())

print("\n=== POST TEXT LENGTH ===")
print(df_a['selftext'].dropna().str.len().describe())

print("\n=== DATE RANGE ===")
print("Earliest:", df_a['created_utc'].min())
print("Latest:  ", df_a['created_utc'].max())

print("\n=== SAMPLE POST ===")
print(df_a['selftext'].dropna().iloc[0][:300])

=== MISSING VALUES ===
Unnamed: 0         0
author             0
created_utc        0
score              0
selftext       54564
subreddit          0
title              8
timestamp          0
dtype: int64

=== SUBREDDIT DISTRIBUTION ===
subreddit
depression                                                                       624561
SuicideWatch                                                                     483048
mentalhealth                                                                     303109
Anxiety                                                                          280038
lonely                                                                           157114
                                                                                  ...  
Is this an okay response you suspect is going through a mental health crisis?         1
I could use some advice about my embarrassing past                                    1
LifesLife's beenLife's been going so well, why do 

In [13]:
# ── Part B exploration ─────────────────────────────
print("=== PART B COLUMNS ===")
print(df_b.columns.tolist())

print("\n=== MISSING VALUES ===")
print(df_b.isnull().sum())

print("\n=== LABEL DISTRIBUTION ===")
print(df_b['Label'].value_counts())

print("\n=== SUBREDDIT IN PART B ===")
if 'subreddit' in df_b.columns:
    print(df_b['subreddit'].value_counts())

print("\n=== SAMPLE ROW ===")
print(df_b.iloc[0])

=== PART B COLUMNS ===
['score', 'selftext', 'subreddit', 'title', 'Label', 'CAT 1']

=== MISSING VALUES ===
score         23
selftext      23
subreddit     23
title         23
Label          0
CAT 1        623
dtype: int64

=== LABEL DISTRIBUTION ===
Label
Drug and Alcohol     223
Early Life           200
Personality          200
Trauma and Stress    200
Name: count, dtype: int64

=== SUBREDDIT IN PART B ===
subreddit
depression      277
Anxiety         187
mentalhealth    151
SuicideWatch    137
lonely           48
Name: count, dtype: int64

=== SAMPLE ROW ===
score                                                      1.0
selftext     Tried to watch this documentary “anxious Ameri...
subreddit                                              Anxiety
title                              Do people get over anxiety?
Label                                         Drug and Alcohol
CAT 1                                                      NaN
Name: 0, dtype: object


In [14]:
# ── Check what needs cleaning ──────────────────────

# 1. How many rows are NOT in the 5 valid subreddits
valid_subreddits = [
    'depression', 'SuicideWatch', 
    'mentalhealth', 'Anxiety', 'lonely'
]
invalid = df_a[~df_a['subreddit'].isin(valid_subreddits)]
print(f"Rows with invalid subreddit: {len(invalid):,}")
print(f"Rows with valid subreddit:   {len(df_a) - len(invalid):,}")

# 2. Check for deleted or removed posts
print(f"\nPosts marked [deleted]: {(df_a['selftext'] == '[deleted]').sum():,}")
print(f"Posts marked [removed]: {(df_a['selftext'] == '[removed]').sum():,}")

# 3. Check for very short posts (less than 10 characters)
short = df_a[df_a['selftext'].notna() & (df_a['selftext'].str.len() < 10)]
print(f"Posts under 10 characters: {len(short):,}")

# 4. Check timestamp format
print(f"\nSample UTC timestamp: {df_a['created_utc'].iloc[0]}")
print(f"Type: {type(df_a['created_utc'].iloc[0])}")

# 5. Part B missing rows
print(f"\nPart B rows to drop (all fields missing): {df_b[df_b['selftext'].isna()].shape[0]}")

Rows with invalid subreddit: 3,710
Rows with valid subreddit:   1,847,870

Posts marked [deleted]: 17,098
Posts marked [removed]: 165,985
Posts under 10 characters: 190,455

Sample UTC timestamp: 1588254956
Type: <class 'numpy.int64'>

Part B rows to drop (all fields missing): 23
